In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Setup Phase-Space Grid (Space x, Frequency xi)
x = np.linspace(-3, 3, 200)
xi = np.linspace(-5, 5, 200)
X, Xi = np.meshgrid(x, xi)

# 2. Define a 2x2 Matrix Symbol A(x, xi)
# A(x, xi) = [ xi^2 + x^2,      x * xi    ]
#            [   x * xi  ,   xi^2 + 1.0   ]
A_11 = Xi**2 + X**2
A_12 = X * Xi
A_21 = X * Xi
A_22 = Xi**2 + 1.0

# 3. Compute Spectral Properties (Eigenvalues & Determinant)
# For a 2x2 matrix, eigenvalues can be solved analytically per grid point
trace = A_11 + A_22
det = (A_11 * A_22) - (A_12 * A_21)
discriminant = np.sqrt(np.maximum(0, (trace / 2) ** 2 - det))

lambda_min = (trace / 2) - discriminant
lambda_max = (trace / 2) + discriminant

# 4. Visualization Suite
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
fig.suptitle(
    "2x2 Matrix Symbol Phase-Space Visualization", fontsize=14, fontweight="bold"
)

# Plot 1: Off-Diagonal Coupling Component A_12(x, xi)
im0 = axes[0, 0].pcolormesh(X, Xi, A_12, cmap="coolwarm", shading="auto")
axes[0, 0].set_title("Off-Diagonal Coupling Component $A_{12}(x, \\xi)$")
axes[0, 0].set_xlabel("Space ($x$)")
axes[0, 0].set_ylabel("Frequency ($\\xi$)")
fig.colorbar(im0, ax=axes[0, 0])

# Plot 2: Symbol Determinant
im1 = axes[0, 1].pcolormesh(X, Xi, det, cmap="viridis", shading="auto")
axes[0, 1].set_title("Matrix Symbol Determinant $\\det(A(x, \\xi))$")
axes[0, 1].set_xlabel("Space ($x$)")
axes[0, 1].set_ylabel("Frequency ($\\xi$)")
fig.colorbar(im1, ax=axes[0, 1])

# Plot 3: Minimum Eigenvalue Contour (Energy/Ellipticity profile)
im2 = axes[1, 0].contourf(X, Xi, lambda_min, levels=15, cmap="magma")
axes[1, 0].set_title("Minimum Eigenvalue $\\lambda_{\\min}(x, \\xi)$")
axes[1, 0].set_xlabel("Space ($x$)")
axes[1, 0].set_ylabel("Frequency ($\\xi$)")
fig.colorbar(im2, ax=axes[1, 0])

# Plot 4: Eigenvalue Slice at fixed x = 1.0
idx_x = np.argmin(np.abs(x - 1.0))
axes[1, 1].plot(xi, lambda_min[:, idx_x], label="$\\lambda_{\\min}(\\xi)$", lw=2)
axes[1, 1].plot(xi, lambda_max[:, idx_x], label="$\\lambda_{\\max}(\\xi)$", lw=2)
axes[1, 1].set_title("Eigenvalue Spectrum Slice at $x = 1.0$")
axes[1, 1].set_xlabel("Frequency ($\\xi$)")
axes[1, 1].set_ylabel("Eigenvalue Magnitude")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
test_matrix_psiop.py
====================
Four non-trivial examples to evaluate the new 2x2 matrix-valued
pseudo-differential operator functionalities of psiop.py
(class ``MatrixPseudoDifferentialOperator``).
"""

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

from psiop import MatrixPseudoDifferentialOperator

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 10,
})


# ======================================================================
# Small utilities
# ======================================================================
def rel_l2_error(a, b):
    a, b = np.asarray(a), np.asarray(b)
    nb = np.linalg.norm(b)
    return float(np.linalg.norm(a - b) / (nb if nb > 0 else 1.0))


def report(name, err, tol=1e-9):
    status = "PASS" if err < tol else "FAIL"
    print(f"  [{status}] {name:<42s} rel. L2 err = {err:.3e}")


def spectral_derivative(u, xg):
    """1D periodic spectral d/dx via FFT."""
    N, dx = len(u), xg[1] - xg[0]
    k = 2.0 * np.pi * np.fft.fftfreq(N, d=dx)
    return np.fft.ifft(1j * k * np.fft.fft(u))


def spectral_derivative_axis(u, axis, grid):
    """Periodic spectral derivative along one axis of a 2D array."""
    N, d = u.shape[axis], grid[1] - grid[0]
    k = 2.0 * np.pi * np.fft.fftfreq(N, d=d)
    shape = [1] * u.ndim
    shape[axis] = N
    return np.fft.ifft(1j * k.reshape(shape) * np.fft.fft(u, axis=axis), axis=axis)


def matrix_frobenius_field(M_expr, syms, grids):
    """Evaluate a sympy matrix of symbols on grids -> Frobenius norm field."""
    fns = [sp.lambdify(syms, M_expr[i, j], "numpy")
           for i in range(M_expr.shape[0]) for j in range(M_expr.shape[1])]
    norm2 = np.zeros(grids[0].shape)
    for f in fns:
        val = np.broadcast_to(np.asarray(f(*grids), dtype=complex), grids[0].shape)
        norm2 += np.abs(val) ** 2
    return np.sqrt(norm2)


# ======================================================================
# Example 1 -- 1D acoustic interface: variable-coefficient hyperbolic system
# ======================================================================
def example_1_acoustic_interface():
    print("\n" + "=" * 78)
    print("Example 1 - 1D acoustic system  P = [[0, c(x) xi], [c(x) xi, 0]], c = 2 + sin(x)")
    print("=" * 78)

    x, xi = sp.symbols("x xi", real=True)
    c = 2 + sp.sin(x)
    P = sp.Matrix([[0, c * xi], [c * xi, 0]])
    op = MatrixPseudoDifferentialOperator(P, [x])     # KN quantization, Peetre backend

    N, L = 512, np.pi
    xg = np.linspace(-L, L, N, endpoint=False)
    dx = xg[1] - xg[0]
    kx = 2.0 * np.pi * np.fft.fftfreq(N, d=dx)

    u1 = np.exp(-6.0 * (xg + 1.2) ** 2)
    u2 = np.exp(-6.0 * (xg - 0.8) ** 2) * np.cos(4.0 * xg)

    v1, v2 = op.apply([u1, u2], xg, kx, freq_window=None, clamp=np.inf)

    # Exact KN reference: Op[c(x) xi] w = c(x) (-i w')
    cv = 2.0 + np.sin(xg)
    report("(P u)_1  vs  -i c(x) u2'", rel_l2_error(v1, -1j * cv * spectral_derivative(u2, xg)))
    report("(P u)_2  vs  -i c(x) u1'", rel_l2_error(v2, -1j * cv * spectral_derivative(u1, xg)))

    # eigen_symbol(): lambda_pm(x, xi) = +/- c(x) |xi| -> hyperbolicity check
    x_sel = np.array([-2.5, -1.0, 0.0, 1.5, 3.0])
    xi_line = np.linspace(-8.0, 8.0, 400)
    eigvals, eigvecs = op.eigen_symbol(x_sel[:, None], xi_line[None, :])
    max_imag = float(np.max(np.abs(eigvals.imag)))
    print(f"  max |Im(lambda_pm)| = {max_imag:.2e}  ->  real spectrum: system hyperbolic")

    # ---- figure ----
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(xg, cv, "b", lw=2)
    axes[0].set_xlabel("x"); axes[0].set_ylabel("c(x)")
    axes[0].set_title("Wave speed c(x) = 2 + sin(x)")

    cmap = plt.get_cmap("tab10")
    for i, x0 in enumerate(x_sel):
        col = cmap(i)
        axes[1].plot(xi_line, eigvals[i, :, 0].real, color=col, lw=1.8,
                     label=f"x = {x0:.1f}")
        axes[1].plot(xi_line, eigvals[i, :, 1].real, color=col, lw=1.8, ls="--")
    axes[1].set_xlabel(r"$\xi$"); axes[1].set_ylabel(r"$\lambda_\pm$")
    axes[1].set_title(r"Eigenvalues $\lambda_\pm(x,\xi) = \pm c(x)|\xi|$")
    axes[1].legend(fontsize=8)

    axes[2].plot(xg, np.abs(u1), "C0", lw=1.2, label=r"$|u_1|$ (input)")
    axes[2].plot(xg, np.abs(u2), "C1", lw=1.2, label=r"$|u_2|$ (input)")
    axes[2].plot(xg, np.abs(v1), "C0", lw=2.2, ls="--", label=r"$|(Pu)_1|$ (output)")
    axes[2].plot(xg, np.abs(v2), "C1", lw=2.2, ls="--", label=r"$|(Pu)_2|$ (output)")
    axes[2].set_xlabel("x"); axes[2].set_ylabel("amplitude")
    axes[2].set_title("apply() on a 2-component field")
    axes[2].legend(fontsize=8)

    fig.suptitle("Example 1 - variable-coefficient 2x2 acoustic operator", y=1.02)
    fig.tight_layout()


# ======================================================================
# Example 2 -- non-commutative symbolic calculus: composition & commutator
# ======================================================================
def example_2_noncommutative_calculus():
    print("\n" + "=" * 78)
    print("Example 2 - matrix composition & commutator (non-commutative calculus)")
    print("=" * 78)

    x, xi = sp.symbols("x xi", real=True)

    # ---- (a) constant-coefficient symbols: composition EXACT at any order ----
    A = sp.Matrix([[xi, 1], [0, xi ** 2]])
    B = sp.Matrix([[xi ** 2, 0], [xi, 1]])
    opA = MatrixPseudoDifferentialOperator(A, [x])
    opB = MatrixPseudoDifferentialOperator(B, [x])

    C_ab = opA.compose_asymptotic(opB, order=3, mode="kn")
    C_ba = opB.compose_asymptotic(opA, order=3, mode="kn")
    C_ref = sp.simplify(A * B)

    diffs = [sp.simplify(C_ab[i, j] - C_ref[i, j]) for i in range(2) for j in range(2)]
    ok_exact = all(d == 0 for d in diffs)
    noncomm = sp.simplify(C_ab - C_ba)
    ok_noncomm = any(sp.simplify(noncomm[i, j]) != 0 for i in range(2) for j in range(2))
    print(f"  compose(A, B, order=3) == A*B exactly : {ok_exact}")
    print(f"  A o B != B o A (matrix non-commutativity) : {ok_noncomm}")
    print("  A*B =")
    sp.pprint(C_ref)

    # ---- (b) variable coefficients: commutator nonzero at order 0 ----
    Pm = sp.Matrix([[0, -sp.I * xi + x], [-sp.I * xi - x, 0]])   # Dirac-type
    Qm = sp.Matrix([[xi ** 2, 0], [0, x ** 2]])
    opP = MatrixPseudoDifferentialOperator(Pm, [x])
    opQ = MatrixPseudoDifferentialOperator(Qm, [x])

    comm0 = sp.simplify(opP.commutator_symbolic(opQ, order=0, mode="kn"))
    comm2 = sp.simplify(opP.commutator_symbolic(opQ, order=2, mode="kn"))
    print("  [P, Q] at order 0 (pure matrix commutator, already nonzero) =")
    sp.pprint(comm0)
    print("  [P, Q] at order 2 (with microlocal derivative corrections) =")
    sp.pprint(comm2)

    # ---- figure: Frobenius norm of the commutator in phase space ----
    Xg, XIg = np.meshgrid(np.linspace(-3, 3, 301), np.linspace(-6, 6, 301), indexing="ij")
    f0 = matrix_frobenius_field(comm0, (x, xi), (Xg, XIg))
    f2 = matrix_frobenius_field(comm2, (x, xi), (Xg, XIg))

    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
    for ax, f, t in zip(axes, (f0, f2),
                        ("order 0 (matrix commutator)", "order 2 (+ derivative terms)")):
        pcm = ax.pcolormesh(Xg, XIg, np.log10(1.0 + f), cmap="magma", shading="auto")
        fig.colorbar(pcm, ax=ax, label=r"$\log_{10}(1+\|[P,Q]\|_F)$")
        ax.set_xlabel("x"); ax.set_ylabel(r"$\xi$")
        ax.set_title(f"matrix commutator, {t}")
    
    # FIX: Matplotlib's mathtext doesn't support \begin{pmatrix}. Use bracket notation instead.
    fig.suptitle(r"Example 2 - Frobenius norm of $[P,Q](x,\xi)$,  "
                 r"$P = [[0, -i\xi+x], [-i\xi-x, 0]]$, "
                 r"$Q=\mathrm{diag}(\xi^2, x^2)$", y=1.02)
    fig.tight_layout()


# ======================================================================
# Example 3 -- Dirac operator with domain-wall mass: avoided crossing
# ======================================================================
def example_3_dirac_domain_wall():
    print("\n" + "=" * 78)
    print("Example 3 - Dirac operator  P = [[m(x), xi], [xi, -m(x)]],  m(x) = tanh(x)")
    print("=" * 78)

    x, xi = sp.symbols("x xi", real=True)
    m = sp.tanh(x)
    Pd = sp.Matrix([[m, xi], [xi, -m]])
    op = MatrixPseudoDifferentialOperator(Pd, [x])

    N, L = 1024, 6.0
    xg = np.linspace(-L, L, N, endpoint=False)
    dx = xg[1] - xg[0]
    kx = 2.0 * np.pi * np.fft.fftfreq(N, d=dx)

    k0 = 6.0
    u1 = np.exp(-(xg + 1.5) ** 2) * np.exp(1j * k0 * xg)
    u2 = 0.6 * np.exp(-(xg - 1.0) ** 2 / 0.5)

    v1, v2 = op.apply([u1, u2], xg, kx, freq_window=None, clamp=np.inf)

    # Exact reference: (Pu)_1 = m u1 - i u2',   (Pu)_2 = -i u1' - m u2
    mv = np.tanh(xg)
    report("(P u)_1  vs  m u1 - i u2'", rel_l2_error(v1, mv * u1 - 1j * spectral_derivative(u2, xg)))
    report("(P u)_2  vs  -i u1' - m u2", rel_l2_error(v2, -1j * spectral_derivative(u1, xg) - mv * u2))

    # Eigenvalues lambda_pm(x, xi) = +/- sqrt(xi^2 + m(x)^2): avoided crossing
    Xg, XIg = np.meshgrid(np.linspace(-4, 4, 300), np.linspace(-6, 6, 250), indexing="ij")
    eigvals, _ = op.eigen_symbol(Xg, XIg)
    lam_plus = eigvals[..., 0].real
    lam_minus = eigvals[..., 1].real
    gap = float(np.min(lam_plus - lam_minus))
    print(f"  minimal spectral gap on the plotted window: {gap:.3e} "
          f"(gap closes at x=0, xi=0 since m(0)=0)")

    # ---- figure ----
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    vmax = max(np.abs(lam_minus).max(), np.abs(lam_plus).max())
    pcm0 = axes[0, 0].pcolormesh(Xg, XIg, lam_minus, cmap="RdBu_r",
                                 vmin=-vmax, vmax=vmax, shading="auto")
    fig.colorbar(pcm0, ax=axes[0, 0], label=r"$\lambda_-$")
    axes[0, 0].plot([0], [0], "kx", ms=10, mew=2)
    axes[0, 0].set_title(r"$\lambda_-(x,\xi)=-\sqrt{\xi^2+m(x)^2}$")

    pcm1 = axes[0, 1].pcolormesh(Xg, XIg, lam_plus, cmap="RdBu_r",
                                 vmin=-vmax, vmax=vmax, shading="auto")
    fig.colorbar(pcm1, ax=axes[0, 1], label=r"$\lambda_+$")
    axes[0, 1].plot([0], [0], "kx", ms=10, mew=2)
    axes[0, 1].set_title(r"$\lambda_+(x,\xi)=+\sqrt{\xi^2+m(x)^2}$")
    for ax in axes[0]:
        ax.set_xlabel("x"); ax.set_ylabel(r"$\xi$")

    axes[1, 0].plot(xg, np.abs(u1), "C0", label=r"$|u_1|$")
    axes[1, 0].plot(xg, np.abs(u2), "C1", label=r"$|u_2|$")
    axes[1, 0].plot(xg, mv, "k--", lw=1, label="m(x)")
    axes[1, 0].set_title("input spinor + mass profile")

    axes[1, 1].plot(xg, np.abs(v1), "C0", label=r"$|(Pu)_1|$")
    axes[1, 1].plot(xg, np.abs(v2), "C1", label=r"$|(Pu)_2|$")
    axes[1, 1].set_title("output spinor after apply()")
    for ax in axes[1]:
        ax.set_xlabel("x"); ax.legend(fontsize=8)

    fig.suptitle("Example 3 - Dirac domain wall: avoided crossing & spinor action", y=1.0)
    fig.tight_layout()


# ======================================================================
# Example 4 -- 2D massless Dirac symbol: Dirac cone, P o P = -Laplacian
# ======================================================================
def example_4_2d_dirac_cone():
    print("\n" + "=" * 78)
    print("Example 4 - 2D massless Dirac  P = [[xi, eta], [eta, -xi]]")
    print("=" * 78)

    x, y = sp.symbols("x y", real=True)
    xi, eta = sp.symbols("xi eta", real=True)
    P2 = sp.Matrix([[xi, eta], [eta, -xi]])
    op = MatrixPseudoDifferentialOperator(P2, [x, y])

    N, L = 128, np.pi
    xg = np.linspace(-L, L, N, endpoint=False)
    yg = np.linspace(-L, L, N, endpoint=False)
    dx, dy = xg[1] - xg[0], yg[1] - yg[0]
    kx = 2.0 * np.pi * np.fft.fftfreq(N, d=dx)
    ky = 2.0 * np.pi * np.fft.fftfreq(N, d=dy)

    X, Y = np.meshgrid(xg, yg, indexing="ij")
    u1 = np.exp(-2.0 * (X ** 2 + Y ** 2))
    u2 = np.exp(-2.0 * ((X - 0.7) ** 2 + (Y + 0.5) ** 2))

    v1, v2 = op.apply([u1, u2], xg, kx, y_grid=yg, ky=ky,
                      freq_window=None, clamp=np.inf)

    # Reference: (Pu)_1 = -i(dx u1 + dy u2),  (Pu)_2 = -i dy u1 + i dx u2
    du1x, du1y = spectral_derivative_axis(u1, 0, xg), spectral_derivative_axis(u1, 1, yg)
    du2x, du2y = spectral_derivative_axis(u2, 0, xg), spectral_derivative_axis(u2, 1, yg)
    report("(P u)_1  vs  -i (dx u1 + dy u2)", rel_l2_error(v1, -1j * (du1x + du2y)))
    report("(P u)_2  vs  -i dy u1 + i dx u2", rel_l2_error(v2, -1j * du1y + 1j * du2x))

    # Symbolic: P o P = (xi^2 + eta^2) I  (massless Dirac squares to Laplacian)
    comp = op.compose_asymptotic(op, order=2, mode="kn")
    target = (xi ** 2 + eta ** 2) * sp.eye(2)
    ok = all(sp.simplify(comp[i, j] - target[i, j]) == 0 for i in range(2) for j in range(2))
    print(f"  P o P == (xi^2 + eta^2) I symbolically (mode='kn') : {ok}")

    # Numerical: apply twice == -Laplacian
    w1, w2 = op.apply([v1, v2], xg, kx, y_grid=yg, ky=ky,
                      freq_window=None, clamp=np.inf)
    KXf, KYf = np.meshgrid(kx, ky, indexing="ij")
    minus_lap = lambda u: -np.fft.ifft2((KXf ** 2 + KYf ** 2) * np.fft.fft2(u))
    report("P(P u1)  vs  -Laplacian u1", rel_l2_error(w1, minus_lap(u1)))
    report("P(P u2)  vs  -Laplacian u2", rel_l2_error(w2, minus_lap(u2)))

    # Variable-coefficient 2D composition at order 1 (KN only)
    V = sp.Matrix([[sp.sin(x), 0], [0, sp.cos(y)]])
    opV = MatrixPseudoDifferentialOperator(V, [x, y])
    comp_v = sp.simplify(op.compose_asymptotic(opV, order=1, mode="kn"))
    print("  P o V at order 1 (non-trivial microlocal correction) =")
    sp.pprint(comp_v)

    # Guard check: 2D Weyl matrix composition is documented as not implemented
    try:
        op.compose_asymptotic(op, order=1, mode="weyl")
        print("  WARNING: 2D Weyl matrix composition unexpectedly succeeded")
    except NotImplementedError:
        print("  guard check OK: 2D Weyl matrix composition raises NotImplementedError")

    # ---- figure 1: the Dirac cone ----
    KXc, KYc = np.meshgrid(np.linspace(-8, 8, 120), np.linspace(-8, 8, 120), indexing="ij")
    eigvals, _ = op.eigen_symbol(0.0, 0.0, KXc, KYc)

    fig = plt.figure(figsize=(12, 5))
    for k, (lam, t, sgn) in enumerate((
            (eigvals[..., 0].real, r"$\lambda_+$", "+"),
            (eigvals[..., 1].real, r"$\lambda_-$", "-"))):
        ax = fig.add_subplot(1, 2, k + 1, projection="3d")
        ax.plot_surface(KXc, KYc, lam, cmap="viridis", alpha=0.9)
        ax.set_xlabel(r"$\xi$"); ax.set_ylabel(r"$\eta$"); ax.set_zlabel(t)
        ax.set_title(rf"{t} = {sgn}\sqrt{{\xi^2+\eta^2}}")
    fig.suptitle("Example 4 - Dirac cone of the 2x2 symbol (eigen_symbol)", y=1.0)
    fig.tight_layout()

    # ---- figure 2: input / output field amplitudes ----
    fig2, axes2 = plt.subplots(2, 2, figsize=(10, 8), sharex=True, sharey=True)
    for ax, F, t in zip(axes2.ravel(), (u1, u2, v1, v2),
                        (r"$|u_1|$", r"$|u_2|$", r"$|(Pu)_1|$", r"$|(Pu)_2|$")):
        pcm = ax.pcolormesh(X, Y, np.abs(F), cmap="inferno", shading="auto")
        fig2.colorbar(pcm, ax=ax)
        ax.set_title(t); ax.set_xlabel("x"); ax.set_ylabel("y")
    fig2.suptitle("Example 4 - apply() of the 2D Dirac symbol on a vector field", y=1.0)
    fig2.tight_layout()

# ======================================================================
# Example 5 -- Hodge decomposition on the 2-torus (0-, 1- and 2-forms)
# ======================================================================
def example_5_hodge_torus():
    from psiop import PseudoDifferentialOperator

    print("\n" + "=" * 78)
    print("Example 5 - Hodge decomposition on T^2: 0-forms --d--> 1-forms --d--> 2-forms")
    print("=" * 78)

    x, y = sp.symbols("x y", real=True)
    xi, eta = sp.symbols("xi eta", real=True)

    # ------------------------------------------------------------------
    # 1. The de Rham complex as matrix symbols (KN: d_x = i Op[xi])
    # ------------------------------------------------------------------
    d0     = sp.Matrix([sp.I * xi, sp.I * eta])          # gradient : 0-forms -> 1-forms (2x1)
    d1     = sp.Matrix([[-sp.I * eta, sp.I * xi]])       # curl     : 1-forms -> 2-forms (1x2)
    d0_adj = sp.Matrix([[-sp.I * xi, -sp.I * eta]])      # -div     : 1-forms -> 0-forms (1x2)
    d1_adj = sp.Matrix([sp.I * eta, -sp.I * xi])         # co-curl  : 2-forms -> 1-forms (2x1)

    ok_dd = sp.simplify(d1 * d0) == sp.zeros(1, 1)
    print(f"  d^2 = d1 o d0 = 0  (chain complex)            : {ok_dd}")

    # Hodge Laplacians at each degree: Delta_k = d delta + delta d
    A_sym = sp.simplify(d0 * d0_adj)        # exact block    (d o delta) on 1-forms, 2x2
    B_sym = sp.simplify(d1_adj * d1)        # co-exact block (delta o d) on 1-forms, 2x2
    Lap0  = sp.simplify(d0_adj * d0)        # on 0-forms (1x1)
    Lap2  = sp.simplify(d1 * d1_adj)        # on 2-forms (1x1)
    Lap1  = sp.simplify(A_sym + B_sym)      # on 1-forms (2x2)
    k2 = xi**2 + eta**2
    print(f"  Delta_0 (0-forms) = {Lap0[0]}")
    print(f"  Delta_2 (2-forms) = {Lap2[0]}")
    print("  Delta_1 (1-forms) =")
    sp.pprint(Lap1)
    ok_lap1 = all(sp.simplify(Lap1[i, j] - k2 * sp.eye(2)[i, j]) == 0
                  for i in range(2) for j in range(2))
    print(f"  Delta_1 == (xi^2+eta^2) I_2                   : {ok_lap1}")

    # ------------------------------------------------------------------
    # 2. Composition identities (constant coefficients => exact products)
    # ------------------------------------------------------------------
    opA = MatrixPseudoDifferentialOperator(A_sym, [x, y])
    opB = MatrixPseudoDifferentialOperator(B_sym, [x, y])

    AA   = opA.compose_asymptotic(opA, order=2, mode="kn")
    BB   = opB.compose_asymptotic(opB, order=2, mode="kn")
    AB   = opA.compose_asymptotic(opB, order=2, mode="kn")
    comm = opA.commutator_symbolic(opB, order=2, mode="kn")

    ok_AA   = all(sp.simplify(AA[i, j] - k2 * A_sym[i, j]) == 0 for i in range(2) for j in range(2))
    ok_BB   = all(sp.simplify(BB[i, j] - k2 * B_sym[i, j]) == 0 for i in range(2) for j in range(2))
    ok_AB   = all(sp.simplify(AB[i, j]) == 0 for i in range(2) for j in range(2))
    ok_comm = all(sp.simplify(comm[i, j]) == 0 for i in range(2) for j in range(2))
    print(f"  A o A == Delta_1 . A  (projector^2 = Delta.A) : {ok_AA}")
    print(f"  B o B == Delta_1 . B                          : {ok_BB}")
    print(f"  A o B == 0  (exact sector ⟂ co-exact sector)  : {ok_AB}")
    print(f"  [A, B] == 0 (the two Hodge sectors commute)   : {ok_comm}")

    # ------------------------------------------------------------------
    # 3. Numerical Hodge decomposition of a 1-form with known answer
    # ------------------------------------------------------------------
    N, L = 128, np.pi
    xg = np.linspace(-L, L, N, endpoint=False)
    yg = np.linspace(-L, L, N, endpoint=False)
    dx, dy = xg[1] - xg[0], yg[1] - yg[0]
    kx = 2.0 * np.pi * np.fft.fftfreq(N, d=dx)
    ky = 2.0 * np.pi * np.fft.fftfreq(N, d=dy)
    X, Y = np.meshgrid(xg, yg, indexing="ij")

    # ground truth: alpha = d(phi0) + delta(psi0) + harmonic
    phi0 = np.sin(X) * np.cos(Y)                          # 0-form potential
    psi0 = np.cos(2 * X) * np.sin(Y)                      # 2-form potential
    cL = (0.3, -0.2)                                      # harmonic (constant) part
    long_true  = [np.cos(X) * np.cos(Y), -np.sin(X) * np.sin(Y)]        # d phi0
    trans_true = [np.cos(2 * X) * np.cos(Y), 2 * np.sin(2 * X) * np.sin(Y)]  # delta psi0
    harm_true  = [np.full_like(X, cL[0]), np.full_like(X, cL[1])]
    alpha = [long_true[i] + trans_true[i] + harm_true[i] for i in range(2)]

    # regularized Hodge projectors: exact on all nonzero modes, kill k=0
    eps = 1e-8
    den = xi**2 + eta**2 + eps
    opPL = MatrixPseudoDifferentialOperator(sp.simplify(A_sym / den), [x, y])
    opPT = MatrixPseudoDifferentialOperator(sp.simplify(B_sym / den), [x, y])

    kw = dict(freq_window=None, clamp=np.inf)
    aL1, aL2 = opPL.apply(alpha, xg, kx, y_grid=yg, ky=ky, **kw)
    aT1, aT2 = opPT.apply(alpha, xg, kx, y_grid=yg, ky=ky, **kw)
    alpha_L, alpha_T = [aL1, aL2], [aT1, aT2]
    alpha_H = [alpha[i] - alpha_L[i] - alpha_T[i] for i in range(2)]

    report("exact part    alpha_L vs d(phi0)",    rel_l2_error(alpha_L, long_true),  tol=1e-6)
    report("co-exact part alpha_T vs delta(psi0)", rel_l2_error(alpha_T, trans_true), tol=1e-6)
    report("harmonic part alpha_H vs constant",   rel_l2_error(alpha_H, harm_true),  tol=1e-6)

    scale = np.linalg.norm(alpha[0])
    harm_osc = float(np.sqrt(sum(np.linalg.norm(alpha_H[i] - alpha_H[i].mean()) ** 2
                                 for i in range(2))))
    print(f"  alpha_H has only the k=0 mode (harmonic): osc/||alpha|| = {harm_osc / scale:.3e}")
    print(f"  recovered harmonic constants: ({alpha_H[0].mean():.4f}, {alpha_H[1].mean():.4f})"
          f"  vs truth {cL}")

    # gauge checks: alpha_L is closed (d alpha_L = 0), alpha_T is co-closed
    curl_aL = (spectral_derivative_axis(alpha_L[1], 0, xg)
               - spectral_derivative_axis(alpha_L[0], 1, yg))
    div_aT  = (spectral_derivative_axis(alpha_T[0], 0, xg)
               + spectral_derivative_axis(alpha_T[1], 1, yg))
    print(f"  ||d(alpha_L)|| / ||alpha||     = {np.linalg.norm(curl_aL) / scale:.3e}  (closed)")
    print(f"  ||delta(alpha_T)|| / ||alpha|| = {np.linalg.norm(div_aT) / scale:.3e}  (co-closed)")

    # recover the 0-form and 2-form potentials (close the complex numerically)
    op_invLap = PseudoDifferentialOperator(1 / den, [x, y], mode='symbol')

    delta_aL = -(spectral_derivative_axis(alpha_L[0], 0, xg)
                 + spectral_derivative_axis(alpha_L[1], 1, yg))
    phi_rec = op_invLap.apply(delta_aL, xg, kx, y_grid=yg, ky=ky, **kw)
    phi_rec = phi_rec - phi_rec.mean()
    dphi = [spectral_derivative_axis(phi_rec, 0, xg),
            spectral_derivative_axis(phi_rec, 1, yg)]
    report("0-form recovered: d(phi_rec) vs alpha_L", rel_l2_error(dphi, alpha_L), tol=1e-6)

    d_aT = (spectral_derivative_axis(alpha_T[1], 0, xg)
            - spectral_derivative_axis(alpha_T[0], 1, yg))
    psi_rec = op_invLap.apply(d_aT, xg, kx, y_grid=yg, ky=ky, **kw)
    psi_rec = psi_rec - psi_rec.mean()
    dpsi = [spectral_derivative_axis(psi_rec, 1, yg),
            -spectral_derivative_axis(psi_rec, 0, xg)]
    report("2-form recovered: delta(psi_rec) vs alpha_T", rel_l2_error(dpsi, alpha_T), tol=1e-6)

    # ------------------------------------------------------------------
    # 4a. figure: spatial Hodge decomposition (quiver plots)
    # ------------------------------------------------------------------
    fig, axes = plt.subplots(2, 2, figsize=(11, 10), sharex=True, sharey=True)
    s = slice(None, None, 6)
    panels = [
        (alpha,   r"input 1-form $\alpha$"),
        (alpha_L, r"exact part $\alpha_L = d\varphi$"),
        (alpha_T, r"co-exact part $\alpha_T = \delta\psi$"),
        (alpha_H, r"harmonic part $\alpha_H$ (constant on $T^2$)"),
    ]
    for ax, (F, t) in zip(axes.ravel(), panels):
        # F[0] and F[1] are complex128 due to the FFT backend.
        # 1. Compute magnitude robustly using complex absolute value
        mag = np.abs(F[0] + 1j * F[1])
        # 2. Extract real parts for the vector components to satisfy quiver/hypot
        F0_real = np.real(F[0])
        F1_real = np.real(F[1])
        
        ax.quiver(X[s, s], Y[s, s], F0_real[s, s], F1_real[s, s], mag[s, s],
                  cmap="coolwarm", pivot="middle")
        ax.set_title(t)
        ax.set_aspect("equal")
        ax.set_xlabel("x"); ax.set_ylabel("y")
    fig.suptitle("Example 5 - Hodge decomposition of a 1-form on the 2-torus", y=0.995)
    fig.tight_layout()

    # ------------------------------------------------------------------
    # 4b. figure: Fourier-space polarization from eigen_symbol
    #     eigenvectors of A are k_hat (exact) and k_hat^perp (co-exact)
    # ------------------------------------------------------------------
    kv = (np.arange(61) - 30) * 0.2 + 0.1          # grid avoiding xi=0 / eta=0
    KXc, KYc = np.meshgrid(kv, kv, indexing="ij")
    eigvals, eigvecs = opA.eigen_symbol(0.0, 0.0, KXc, KYc)
    lam_exact = eigvals[..., 0].real               # = xi^2 + eta^2
    v1 = np.real(eigvecs[..., :, 0])               # should be ∥ k
    v2 = np.real(eigvecs[..., :, 1])               # should be ⟂ k

    r = np.sqrt(KXc**2 + KYc**2)
    khat = np.stack([KXc / r, KYc / r], axis=-1)
    al1_signed = np.sum(v1 * khat, axis=-1)
    v1 = v1 * np.where(al1_signed < 0, -1.0, 1.0)[..., None]   # fix ± ambiguity
    al1 = np.abs(al1_signed)
    al2 = np.abs(np.sum(v2 * khat, axis=-1))
    print(f"  polarization: max |v_exact . k_hat - 1| = {np.max(np.abs(al1 - 1)):.2e}")
    print(f"  polarization: max |v_coexact . k_hat|   = {np.max(al2):.2e}  (orthogonality)")

    fig2, ax2 = plt.subplots(2, 2, figsize=(11, 10))
    pcm = ax2[0, 0].pcolormesh(KXc, KYc, lam_exact, cmap="viridis", shading="auto")
    fig2.colorbar(pcm, ax=ax2[0, 0])
    ax2[0, 0].set_title(r"eigenvalue $|k|^2$ of the exact block $A$")

    qq = slice(None, None, 2)
    ax2[0, 1].quiver(KXc[qq, qq], KYc[qq, qq], v1[qq, qq, 0], v1[qq, qq, 1],
                     pivot="middle", color="C0")
    ax2[0, 1].set_title(r"eigenvector $v_L \parallel k$ (exact sector)")

    ax2[1, 0].quiver(KXc[qq, qq], KYc[qq, qq], v2[qq, qq, 0], v2[qq, qq, 1],
                     pivot="middle", color="C3")
    ax2[1, 0].set_title(r"eigenvector $v_T \perp k$ (co-exact sector)")

    pcm2 = ax2[1, 1].pcolormesh(KXc, KYc, al1, cmap="inferno",
                                vmin=0.999, vmax=1.0, shading="auto")
    fig2.colorbar(pcm2, ax=ax2[1, 1])
    ax2[1, 1].set_title(r"alignment $|v_L \cdot \hat{k}|$")
    for axx in ax2.ravel():
        axx.set_xlabel(r"$\xi$"); axx.set_ylabel(r"$\eta$"); axx.set_aspect("equal")
    fig2.suptitle("Example 5 - Hodge projectors: eigen_symbol polarization in Fourier space",
                  y=0.995)
    fig2.tight_layout()

# ======================================================================
# Example 6 -- Dirichlet-type Hodge decomposition on a bounded square
# ======================================================================
def example_6_hodge_dirichlet_manufactured():
    import scipy.sparse as sparse
    import scipy.sparse.linalg as spla

    print("\n" + "=" * 78)
    print("Example 6 - Dirichlet-type Hodge decomposition on (0, pi)^2")
    print("=" * 78)

    # ------------------------------------------------------------------
    # Symbolic part: same de Rham complex identities as in Example 5.
    # ------------------------------------------------------------------
    x, y = sp.symbols("x y", real=True)
    xi, eta = sp.symbols("xi eta", real=True)

    d0     = sp.Matrix([sp.I * xi, sp.I * eta])
    d1     = sp.Matrix([[-sp.I * eta, sp.I * xi]])
    d0_adj = sp.Matrix([[-sp.I * xi, -sp.I * eta]])
    d1_adj = sp.Matrix([sp.I * eta, -sp.I * xi])

    ok_dd = sp.simplify(d1 * d0) == sp.zeros(1, 1)
    print(f"  d^2 = d1 o d0 = 0 : {ok_dd}")

    A_sym = sp.simplify(d0 * d0_adj)
    B_sym = sp.simplify(d1_adj * d1)

    opA = MatrixPseudoDifferentialOperator(A_sym, [x, y])
    opB = MatrixPseudoDifferentialOperator(B_sym, [x, y])

    AB = opA.compose_asymptotic(opB, order=1, mode="kn")
    ok_AB = all(sp.simplify(AB[i, j]) == 0 for i in range(2) for j in range(2))
    print(f"  A o B == 0 (exact/co-exact orthogonality) : {ok_AB}")

    # ------------------------------------------------------------------
    # Numerical Dirichlet problem on (0, pi)^2.
    #
    # We manufacture:
    #
    #   alpha = d(phi0) + delta(psi0)
    #
    # with phi0 = psi0 = 0 on the boundary.
    # ------------------------------------------------------------------
    N = 80
    L = np.pi
    h = L / (N + 1)

    xg = np.arange(1, N + 1) * h
    yg = np.arange(1, N + 1) * h
    X, Y = np.meshgrid(xg, yg, indexing="ij")

    # Dirichlet potentials, zero on the boundary
    phi0 = np.sin(X) * np.sin(Y)
    psi0 = np.sin(2 * X) * np.sin(Y)

    # Exact part: d(phi0) = grad(phi0)
    exact0 = [
        np.cos(X) * np.sin(Y),
        np.sin(X) * np.cos(Y),
    ]

    # Co-exact part: delta(psi0) = (dy psi0, -dx psi0)
    coex0 = [
        np.sin(2 * X) * np.cos(Y),
        -2 * np.cos(2 * X) * np.sin(Y),
    ]

    alpha = [
        exact0[0] + coex0[0],
        exact0[1] + coex0[1],
    ]

    # For this manufactured field:
    #
    #   div(alpha) = Delta(phi0) = -2 phi0
    #   curl(alpha) = -Delta(psi0) = 5 psi0
    #
    div_alpha = -2.0 * phi0
    curl_alpha = 5.0 * psi0

    # ------------------------------------------------------------------
    # Simple 5-point finite-difference Dirichlet Poisson solver.
    #
    # Solves:
    #
    #   Delta u = f in Omega,
    #   u = 0 on partial Omega.
    # ------------------------------------------------------------------
    def poisson_dirichlet_fd(f, length):
        n = f.shape[0]
        hh = length / (n + 1)

        T = sparse.diags(
            [1.0, -2.0, 1.0],
            [-1, 0, 1],
            shape=(n, n),
            format="csr",
        ) / hh**2

        I = sparse.eye(n, format="csr")

        A = sparse.kron(T, I, format="csr") + sparse.kron(I, T, format="csr")

        return spla.spsolve(A, f.ravel()).reshape(n, n)

    # Recover potentials from Dirichlet Poisson problems:
    #
    #   Delta phi = div(alpha)
    #   Delta psi = -curl(alpha)
    #
    phi_rec = poisson_dirichlet_fd(div_alpha, L)
    psi_rec = poisson_dirichlet_fd(-curl_alpha, L)

    report("Dirichlet Poisson: phi_rec vs phi0",
           rel_l2_error(phi_rec, phi0), tol=1e-2)

    report("Dirichlet Poisson: psi_rec vs psi0",
           rel_l2_error(psi_rec, psi0), tol=1e-2)

    # ------------------------------------------------------------------
    # Reconstruct vector fields from the recovered potentials.
    #
    # We pad with zeros because the Dirichlet boundary values are zero.
    # ------------------------------------------------------------------
    def grad_dirichlet(u, hh):
        up = np.pad(u, 1, mode="constant", constant_values=0.0)

        ux = (up[2:, 1:-1] - up[:-2, 1:-1]) / (2 * hh)
        uy = (up[1:-1, 2:] - up[1:-1, :-2]) / (2 * hh)

        return ux, uy

    phix, phiy = grad_dirichlet(phi_rec, h)
    psix, psiy = grad_dirichlet(psi_rec, h)

    exact_rec = [phix, phiy]
    coex_rec = [psiy, -psix]

    resid = [
        alpha[0] - exact_rec[0] - coex_rec[0],
        alpha[1] - exact_rec[1] - coex_rec[1],
    ]

    report("Dirichlet exact part d(phi_rec)",
           rel_l2_error(exact_rec, exact0), tol=2e-2)

    report("Dirichlet co-exact part delta(psi_rec)",
           rel_l2_error(coex_rec, coex0), tol=2e-2)

    resid_norm = float(np.sqrt(sum(np.linalg.norm(r) ** 2 for r in resid)))
    alpha_norm = float(np.sqrt(sum(np.linalg.norm(a) ** 2 for a in alpha)))

    report("Dirichlet harmonic remainder",
           resid_norm / alpha_norm, tol=2e-2)

    # ------------------------------------------------------------------
    # Figure: recovered potentials and vector fields.
    # ------------------------------------------------------------------
    fig, axes = plt.subplots(2, 2, figsize=(11, 9))

    pcm0 = axes[0, 0].pcolormesh(X, Y, phi_rec, cmap="RdBu_r", shading="auto")
    fig.colorbar(pcm0, ax=axes[0, 0])
    axes[0, 0].set_title(r"Recovered Dirichlet 0-form $\varphi$")
    axes[0, 0].set_xlabel("x")
    axes[0, 0].set_ylabel("y")

    pcm1 = axes[0, 1].pcolormesh(X, Y, psi_rec, cmap="RdBu_r", shading="auto")
    fig.colorbar(pcm1, ax=axes[0, 1])
    axes[0, 1].set_title(r"Recovered Dirichlet 2-form potential $\psi$")
    axes[0, 1].set_xlabel("x")
    axes[0, 1].set_ylabel("y")

    s = slice(None, None, 4)

    mag_in = np.hypot(alpha[0], alpha[1])
    axes[1, 0].quiver(
        X[s, s], Y[s, s],
        alpha[0][s, s], alpha[1][s, s],
        mag_in[s, s],
        cmap="coolwarm",
        pivot="middle",
    )
    axes[1, 0].set_title(r"Input 1-form $\alpha$")
    axes[1, 0].set_xlabel("x")
    axes[1, 0].set_ylabel("y")
    axes[1, 0].set_aspect("equal")

    mag_res = np.hypot(resid[0], resid[1])
    axes[1, 1].quiver(
        X[s, s], Y[s, s],
        resid[0][s, s], resid[1][s, s],
        mag_res[s, s],
        cmap="coolwarm",
        pivot="middle",
    )
    axes[1, 1].set_title(r"Harmonic remainder $h$")
    axes[1, 1].set_xlabel("x")
    axes[1, 1].set_ylabel("y")
    axes[1, 1].set_aspect("equal")

    fig.suptitle(
        "Example 6 - Dirichlet-type Hodge decomposition on a bounded square",
        y=0.995,
    )
    fig.tight_layout()


# ======================================================================
if __name__ == "__main__":
    example_1_acoustic_interface()
    example_2_noncommutative_calculus()
    example_3_dirac_domain_wall()
    example_4_2d_dirac_cone()
    example_5_hodge_torus()
    example_6_hodge_dirichlet_manufactured()
    print("\nAll examples executed. Close the figure windows to exit.")
    plt.show()